# 1. Data Processing and Visualizations


In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
import json
import os
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')


## Text Cleaning Function


In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))
PRIORITY_TERMS = r'\b(dear|support|team|ni|hello|hope|well|customer|data|please|message|team|nan|could|would|assistance|problem|issue|failure|system|update)\b'
HTML_ARTIFACTS = r'\b(href|src|width|height|font|size|face|arial|sans|serif|color|border|style|nbsp|img|align|center|br|div|table|tr|td|span|strong|em)\b'

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(HTML_ARTIFACTS, '', text)
    text = re.sub(r'[\r\n\t\a\b]+', ' ', text)
    text = re.sub(PRIORITY_TERMS, '', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words and len(w) > 1]
    return ' '.join(tokens)


## Load and Combine Datasets


In [ ]:
# Load Data
a = pd.read_csv('../raw_data_sets/aa_dataset-tickets-multi-lang-5-2-50-version.csv')
b = pd.read_csv('../raw_data_sets/dataset-tickets-multi-lang-4-20k.csv')
c = pd.read_csv('../raw_data_sets/email_dataset.csv')

a = a[a['language'] == 'en']
b = b[b['language'] == 'en']

base_cols = ['type', 'priority', 'body', 'subject']
tag_cols = [f'tag_{i}' for i in range(1, 9)]
req_cols = base_cols + tag_cols
a_sel = a[[col for col in req_cols if col in a.columns]]
b_sel = b[[col for col in req_cols if col in b.columns]]
tickets = pd.concat([a_sel, b_sel], ignore_index=True)

def process_type_row(row):
    curr = row.get('type', '')
    if curr in ['Incident', 'Change']:
        tags = [str(row.get(f'tag_{i}', '')).lower() for i in range(1, 9)]
        if 'feedback' in tags: return 'Feedback'
    if curr == 'Problem': return 'Complaint'
    return curr

tickets['type'] = tickets.apply(process_type_row, axis=1)
valid_types = ['Request', 'Complaint', 'Feedback']
tickets = tickets[tickets['type'].isin(valid_types)].copy()

c = c.rename(columns={'Label': 'type', 'Body': 'body', 'Subject': 'subject'})
c = c[c['type'] == 'spam'].copy()
c['priority'] = 'low'
for t in tag_cols: c[t] = np.nan

master = pd.concat([tickets[base_cols], c[base_cols]], ignore_index=True)
master.dropna(subset=['body', 'type', 'priority'], inplace=True)

min_cnt = master['type'].value_counts().min()
balanced = master.groupby('type', group_keys=False).apply(lambda x: x.sample(min_cnt, random_state=42)).reset_index(drop=True)
balanced = balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print('Data Loading Complete. Shape:', balanced.shape)
print(balanced['type'].value_counts())


## Visualizations


In [ ]:
fig = px.bar(balanced['type'].value_counts().reset_index(), x='type', y='count', title='Distribution of Email Types', color='type')
fig.show()

fig2 = px.bar(balanced['priority'].value_counts().reset_index(), x='priority', y='count', title='Distribution of Urgency', color='priority')
fig2.show()

balanced['body_length'] = balanced['body'].apply(lambda x: len(str(x).split()))
fig3 = px.histogram(balanced, x='body_length', color='type', title='Body Length Distribution by Type', nbins=50, barmode='overlay')
fig3.show()


## Clean and Save


In [ ]:
print('Cleaning Subject and Body...')
balanced['body_cleaned'] = balanced['body'].apply(clean_text)
balanced['subject_cleaned'] = balanced['subject'].apply(clean_text)

import os
os.makedirs('../cleaned_dataset-folder', exist_ok=True)
balanced.to_csv('../cleaned_dataset-folder/processed_dataset.csv', index=False)
print('Dataset saved.')
